# 1. Integer solution via Pyomo

Lecture: *Week 5 - Mixed Integer Programming*.

*Original script: `5 Source Files/Python/intlinprog_pyomo.py`. Runs in the browser; Pyomo models use HiGHS instead of GLPK with the same answers.*

In [ ]:
# Run this cell first.
# In the browser (JupyterLite) it installs the extra packages into this tab;
# in Colab or Anaconda it installs them with pip (already installed = no change).
%pip install pyomo highspy

import sys
if sys.platform == "emscripten":  # only in the browser (JupyterLite): adapt Pyomo to it
    import contextlib, pyomo.common.tee, pyomo.common.formatting as _fmt
    # 1) browsers have no threads: skip Pyomo's threaded capture of solver output
    pyomo.common.tee.capture_output = lambda *args, **kwargs: contextlib.nullcontext()
    for _m in ("pyomo.contrib.appsi.solvers.highs", "pyomo.contrib.solver.solvers.highs"):
        if _m in sys.modules:
            sys.modules[_m].capture_output = pyomo.common.tee.capture_output
    # 2) the notebook's output returns None from write(); model.pprint()/display() need a number
    if not getattr(_fmt.StreamIndenter, "_browser_patched", False):
        class _Counted:
            def __init__(self, s): self._s = s
            def write(self, data): self._s.write(data); return len(data)
            def __getattr__(self, name): return getattr(self._s, name)
        def _init(self, ostream, indent=" " * 4, _orig=_fmt.StreamIndenter.__init__):
            _orig(self, ostream, indent)
            if not isinstance(self.target_os, _Counted):
                object.__setattr__(self, "target_os", _Counted(self.target_os))
        _fmt.StreamIndenter.__init__ = _init
        _fmt.StreamIndenter._browser_patched = True

In [ ]:
from pyomo.environ import *

In [ ]:
# Create a concrete model
model = ConcreteModel()

In [ ]:
# Define the integer decision variables with lower and upper bounds
model.x = Var([0, 1], domain=NonNegativeIntegers)

In [ ]:
# Define the objective function coefficients as a list
f = [2, 3.5]

In [ ]:
# Define the objective function
model.obj = Objective(expr=sum(f[i] * model.x[i] for i in [0, 1]), sense=maximize)

In [ ]:
# Add the constraints to the model
model.c1 = Constraint(expr=3 * model.x[0] + 2 * model.x[1] <= 14)
model.c2 = Constraint(expr=2 * model.x[0] + 3 * model.x[1] <= 12.5)
model.c3 = Constraint(expr=-model.x[0] + model.x[1] <= 1)

In [ ]:
# Create a solver
solver = SolverFactory('appsi_highs')

In [ ]:
# Display the model
model.pprint()

In [ ]:
# Solve the model
solver.solve(model, tee=False)  # Set tee=True to display solver output

In [ ]:
# Display the results
model.display()